# Práctica 02 — Reproducibilidad

**Curso:** MLOps y Monitorización con Python · **Clase 02** · **Tiempo estimado:** 40-55 minutos
**Notebook de clase:** `notebooks_clase/clase02_reproducibilidad.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio parte de un fragmento de código con **el mismo tipo de problema** que corregiste en el notebook de clase, sobre datos y modelos pequeños de juguete.
- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `pandas`, `PyYAML` y `scikit-learn` (`RandomForestClassifier`, `train_test_split`).

---
## Celda 0 — Setup

In [ ]:
import os
import inspect
import tempfile

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

print("Librerías cargadas correctamente.")

## Ejercicio 1 — Una línea de `requirements.txt`

`linea_requirements` debe devolver una línea con el formato exacto de `requirements.txt` (`paquete==version`), a partir de dos parámetros. Complétala.

In [ ]:
def linea_requirements(paquete, version):
    return paquete      # TODO: devuelve f"{paquete}=={version}"

# Comprobación automática
assert linea_requirements("numpy", "2.2.6") == "numpy==2.2.6", "Completa las líneas marcadas con TODO"
assert linea_requirements("pandas", "3.0.6") == "pandas==3.0.6", "Completa las líneas marcadas con TODO"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Generar datos sin tocar el estado global

`generar_datos_reproducible` usa `np.random.rand` directamente (el estado global de NumPy), igual que la celda 4 del notebook sucio. Complétala para que use un generador propio, `np.random.default_rng(seed)`, y así dos llamadas con la misma `seed` den el mismo resultado.

In [ ]:
def generar_datos_reproducible(n, seed):
    return np.random.rand(n)      # TODO: usa rng = np.random.default_rng(seed) y rng.random(n)

# Comprobación automática
a = generar_datos_reproducible(10, seed=1)
b = generar_datos_reproducible(10, seed=1)
assert np.array_equal(a, b), "Completa las líneas marcadas con TODO"
c = generar_datos_reproducible(10, seed=2)
assert not np.array_equal(a, c), "con semillas distintas el resultado también debería ser distinto"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Cargar la configuración desde un archivo YAML

`cargar_config` debe leer un archivo YAML con `yaml.safe_load` y devolver el diccionario resultante. De momento no hace nada. Complétala.

In [ ]:
def cargar_config(ruta):
    return {}      # TODO: abre «ruta» y devuelve yaml.safe_load(archivo)

# Comprobación automática
config_prueba = {"n_filas": 100, "semillas": {"datos": 1, "split": 2, "modelo": 3}}
carpeta_temp = tempfile.mkdtemp()
ruta_temp = os.path.join(carpeta_temp, "config.yaml")
with open(ruta_temp, "w") as f:
    yaml.safe_dump(config_prueba, f)
resultado = cargar_config(ruta_temp)
assert resultado == config_prueba, "Completa las líneas marcadas con TODO"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Un modelo con todos los parámetros en `config`

`entrenar_modelo_configurable` crea un `RandomForestClassifier` con `n_estimators` y `random_state` escritos directamente en el código, ignorando `config`. Complétala para que tome `n_estimators`, `max_depth` y `random_state` (la semilla del modelo) desde `config["hiperparametros"]` y `config["semillas"]["modelo"]`.

In [ ]:
def entrenar_modelo_configurable(X, y, config):
    modelo = RandomForestClassifier(n_estimators=10, random_state=0)      # TODO: usa config["hiperparametros"] y config["semillas"]["modelo"]
    modelo.fit(X, y)
    return modelo

# Comprobación automática
config_prueba = {"hiperparametros": {"n_estimators": 15, "max_depth": 3}, "semillas": {"modelo": 5}}
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
m1 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert m1.get_params()["n_estimators"] == 15, "Completa las líneas marcadas con TODO"
assert m1.get_params()["max_depth"] == 3, "Completa las líneas marcadas con TODO"
assert m1.get_params()["random_state"] == 5, "Completa las líneas marcadas con TODO"
m2 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert list(m1.predict(X_juguete)) == list(m2.predict(X_juguete)), "con la misma config las predicciones deben coincidir"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Crear la estructura mínima de carpetas

`crear_estructura` debe crear, dentro de `base`, las carpetas `src`, `config` y `tests` (con `os.makedirs(..., exist_ok=True)`, para que no falle si ya existen) y devolver sus nombres ordenados alfabéticamente.

In [ ]:
def crear_estructura(base):
    return []      # TODO: crea src/, config/ y tests/ bajo «base», devuelve sorted(["src", "config", "tests"])

# Comprobación automática
base_temp = tempfile.mkdtemp()
resultado = crear_estructura(base_temp)
assert resultado == ["config", "src", "tests"], "Completa las líneas marcadas con TODO"
for carpeta in ("src", "config", "tests"):
    assert os.path.isdir(os.path.join(base_temp, carpeta)), "falta la carpeta " + carpeta
crear_estructura(base_temp)      # no debe fallar si ya existen
print("Ejercicio 5 correcto ✔")

---
## Reto final — una `pipeline` que integra datos, config y modelo

Completa `pipeline_reproducible_practica(config)`, que debe:

- generar los datos con `generar_datos_reproducible(config["n_filas"], config["semillas"]["datos"])` (ejercicio 2),
- dividir con `train_test_split(X, y, test_size=config["test_size"], random_state=config["semillas"]["split"])`,
- entrenar con `entrenar_modelo_configurable(X_train, y_train, config)` (ejercicio 4),
- devolver `(modelo, exactitud, predicciones)` sobre el conjunto de prueba.

`X` es un `DataFrame` de una sola columna (`pd.DataFrame({"x": datos})`) e `y` son las etiquetas: `(datos > 0.5).astype(int)`.

In [ ]:
def pipeline_reproducible_practica(config):
    datos = None      # TODO: generar_datos_reproducible(config["n_filas"], config["semillas"]["datos"])
    X = pd.DataFrame({"x": datos})
    y = (datos > 0.5).astype(int)
    X_train, X_test, y_train, y_test = None, None, None, None      # TODO: train_test_split con test_size y random_state de config
    modelo = None      # TODO: entrenar_modelo_configurable(X_train, y_train, config)
    exactitud = None      # TODO: modelo.score(X_test, y_test)
    predicciones = None      # TODO: modelo.predict(X_test)
    return modelo, exactitud, predicciones

# Comprobación automática
config_reto = {
    "n_filas": 60, "test_size": 0.25,
    "hiperparametros": {"n_estimators": 10, "max_depth": 3},
    "semillas": {"datos": 11, "split": 11, "modelo": 11},
}
resultado = pipeline_reproducible_practica(config_reto)
assert resultado is not None, "Completa las líneas marcadas con TODO"
modelo, exactitud, predicciones = resultado
assert hasattr(modelo, "predict"), "devuelve el modelo entrenado, no None"
assert isinstance(exactitud, float), "devuelve la exactitud como número"
modelo_b, exactitud_b, predicciones_b = pipeline_reproducible_practica(config_reto)
assert exactitud == exactitud_b, "con la misma config la exactitud debe repetirse"
assert list(predicciones) == list(predicciones_b), "con la misma config las predicciones deben repetirse"
print("Reto final correcto ✔")

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def linea_requirements(paquete, version):
    return f"{paquete}=={version}"

# Comprobación automática
assert linea_requirements("numpy", "2.2.6") == "numpy==2.2.6", "Completa las líneas marcadas con TODO"
assert linea_requirements("pandas", "3.0.6") == "pandas==3.0.6", "Completa las líneas marcadas con TODO"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def generar_datos_reproducible(n, seed):
    rng = np.random.default_rng(seed)
    return rng.random(n)

# Comprobación automática
a = generar_datos_reproducible(10, seed=1)
b = generar_datos_reproducible(10, seed=1)
assert np.array_equal(a, b), "Completa las líneas marcadas con TODO"
c = generar_datos_reproducible(10, seed=2)
assert not np.array_equal(a, c), "con semillas distintas el resultado también debería ser distinto"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def cargar_config(ruta):
    with open(ruta) as archivo:
        return yaml.safe_load(archivo)

# Comprobación automática
config_prueba = {"n_filas": 100, "semillas": {"datos": 1, "split": 2, "modelo": 3}}
carpeta_temp = tempfile.mkdtemp()
ruta_temp = os.path.join(carpeta_temp, "config.yaml")
with open(ruta_temp, "w") as f:
    yaml.safe_dump(config_prueba, f)
resultado = cargar_config(ruta_temp)
assert resultado == config_prueba, "Completa las líneas marcadas con TODO"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def entrenar_modelo_configurable(X, y, config):
    modelo = RandomForestClassifier(
        n_estimators=config["hiperparametros"]["n_estimators"],
        max_depth=config["hiperparametros"]["max_depth"],
        random_state=config["semillas"]["modelo"],
    )
    modelo.fit(X, y)
    return modelo

# Comprobación automática
config_prueba = {"hiperparametros": {"n_estimators": 15, "max_depth": 3}, "semillas": {"modelo": 5}}
X_juguete = np.arange(40).reshape(20, 2)
y_juguete = np.array([0, 1] * 10)
m1 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert m1.get_params()["n_estimators"] == 15, "Completa las líneas marcadas con TODO"
assert m1.get_params()["max_depth"] == 3, "Completa las líneas marcadas con TODO"
assert m1.get_params()["random_state"] == 5, "Completa las líneas marcadas con TODO"
m2 = entrenar_modelo_configurable(X_juguete, y_juguete, config_prueba)
assert list(m1.predict(X_juguete)) == list(m2.predict(X_juguete)), "con la misma config las predicciones deben coincidir"
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def crear_estructura(base):
    carpetas = ["src", "config", "tests"]
    for carpeta in carpetas:
        os.makedirs(os.path.join(base, carpeta), exist_ok=True)
    return sorted(carpetas)

# Comprobación automática
base_temp = tempfile.mkdtemp()
resultado = crear_estructura(base_temp)
assert resultado == ["config", "src", "tests"], "Completa las líneas marcadas con TODO"
for carpeta in ("src", "config", "tests"):
    assert os.path.isdir(os.path.join(base_temp, carpeta)), "falta la carpeta " + carpeta
crear_estructura(base_temp)      # no debe fallar si ya existen
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def pipeline_reproducible_practica(config):
    datos = generar_datos_reproducible(config["n_filas"], config["semillas"]["datos"])
    X = pd.DataFrame({"x": datos})
    y = (datos > 0.5).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config["test_size"], random_state=config["semillas"]["split"])
    modelo = entrenar_modelo_configurable(X_train, y_train, config)
    exactitud = modelo.score(X_test, y_test)
    predicciones = modelo.predict(X_test)
    return modelo, exactitud, predicciones

# Comprobación automática
config_reto = {
    "n_filas": 60, "test_size": 0.25,
    "hiperparametros": {"n_estimators": 10, "max_depth": 3},
    "semillas": {"datos": 11, "split": 11, "modelo": 11},
}
resultado = pipeline_reproducible_practica(config_reto)
assert resultado is not None, "Completa las líneas marcadas con TODO"
modelo, exactitud, predicciones = resultado
assert hasattr(modelo, "predict"), "devuelve el modelo entrenado, no None"
assert isinstance(exactitud, float), "devuelve la exactitud como número"
modelo_b, exactitud_b, predicciones_b = pipeline_reproducible_practica(config_reto)
assert exactitud == exactitud_b, "con la misma config la exactitud debe repetirse"
assert list(predicciones) == list(predicciones_b), "con la misma config las predicciones deben repetirse"
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (dependencias bloqueadas, semillas, estructura de carpetas, separar la configuración)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*